In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import simpy
from LEOEnvironmentRL import initialize, load_route_from_csv  # Use RL version
import pandas as pd
import os
from stable_baselines3 import DQN
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
import torch
import random
import matplotlib.pyplot as plt
import plotly.graph_objects as go
# %% 
import sb3_contrib
from HandoverEnvironment import LEOEnv as LEOEnvPPO 
from HandoverEnvironment import mask_fn, predict_valid_action
from HandoverEnvironment_DQN import LEOEnv as LEOEnvDQN
from HandoverEnvironment_DQN import predict_valid_action as predict_valid_action_dqn
from HandoverEnvironment_ODT import LEOEnv as LEOEnvODT
from HandoverEnvironment_ODT import predict_valid_action_dt
from ODT import OnlineDecisionTransformer
from LEOEnvironment import LEOEnv as LEOEnvBase

['ARS', 'CrossQ', 'MaskablePPO', 'QRDQN', 'RecurrentPPO', 'TQC', 'TRPO', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'ars', 'common', 'crossq', 'file_handler', 'os', 'ppo_mask', 'ppo_recurrent', 'qrdqn', 'tqc', 'trpo', 'version_file']


In [2]:
import os

scenarios = ["no_scenario", "demand_aware", "multi_objective", "peak_hour", "large_aircraft"]
service_drop_idx = 16

for scenario in scenarios:
    base_path = f"BASELINE_observations_{scenario}.csv"
    ppo_path = f"PPO_observations_{scenario}.csv"
    dqn_path = f"DQN_observations_{scenario}.csv"
    odt_path = f"ODT_observations_{scenario}.csv"
    odt_finetuned_path = f"ODT_FINETUNED_observations_{scenario}.csv"

    if not (os.path.exists(base_path) and os.path.exists(ppo_path) and os.path.exists(dqn_path) and os.path.exists(odt_path) and os.path.exists(odt_finetuned_path)):
        print(f"Missing files for scenario '{scenario}', skipping.")
        continue

    obs_base = pd.read_csv(base_path).values
    obs_ppo = pd.read_csv(ppo_path).values
    obs_dqn = pd.read_csv(dqn_path).values
    obs_odt = pd.read_csv(odt_path).values
    obs_odt_finetuned = pd.read_csv(odt_finetuned_path).values

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=np.cumsum(obs_base[:, service_drop_idx]), mode='lines', name='Baseline'))
    fig.add_trace(go.Scatter(y=np.cumsum(obs_ppo[:, service_drop_idx]), mode='lines', name='PPO'))
    fig.add_trace(go.Scatter(y=np.cumsum(obs_dqn[:, service_drop_idx]), mode='lines', name='DQN'))
    fig.add_trace(go.Scatter(y=np.cumsum(obs_odt[:, service_drop_idx]), mode='lines', name='ODT'))
    fig.add_trace(go.Scatter(y=np.cumsum(obs_odt_finetuned[:, service_drop_idx]), mode='lines', name='ODT Finetuned'))
    fig.update_layout(
        title=f"Cumulative Service Drop Over Time - {scenario}",
        xaxis_title='Step',
        yaxis_title='Total Service Drop (s)',
        legend_title='Agent',
        template='plotly_white'
    )
    fig.show()


In [3]:
import os

scenarios = ["no_scenario", "demand_aware", "multi_objective", "peak_hour", "large_aircraft"]
handover_idx = 6

for scenario in scenarios:
    base_path = f"BASELINE_observations_{scenario}.csv"
    ppo_path = f"PPO_observations_{scenario}.csv"
    dqn_path = f"DQN_observations_{scenario}.csv"
    odt_path = f"ODT_observations_{scenario}.csv"
    odt_finetuned_path = f"ODT_FINETUNED_observations_{scenario}.csv"

    if not (os.path.exists(base_path) and os.path.exists(ppo_path) and os.path.exists(dqn_path) and os.path.exists(odt_path) and os.path.exists(odt_finetuned_path)):
        print(f"Missing files for scenario '{scenario}', skipping.")
        continue

    obs_base = pd.read_csv(base_path).values
    obs_ppo = pd.read_csv(ppo_path).values
    obs_dqn = pd.read_csv(dqn_path).values
    obs_odt = pd.read_csv(odt_path).values
    obs_odt_finetuned = pd.read_csv(odt_finetuned_path).values

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=np.cumsum(obs_base[:, handover_idx]), mode='lines', name='Baseline'))
    fig.add_trace(go.Scatter(y=np.cumsum(obs_ppo[:, handover_idx]), mode='lines', name='PPO'))
    fig.add_trace(go.Scatter(y=np.cumsum(obs_dqn[:, handover_idx]), mode='lines', name='DQN'))
    fig.add_trace(go.Scatter(y=np.cumsum(obs_odt[:, handover_idx]), mode='lines', name='ODT'))
    fig.add_trace(go.Scatter(y=np.cumsum(obs_odt_finetuned[:, handover_idx]), mode='lines', name='ODT Finetuned'))
    fig.update_layout(
        title=f"Total Handovers Over Time - {scenario}",
        xaxis_title='Step',
        yaxis_title='Total Handovers',
        legend_title='Agent',
        template='plotly_white'
    )
    fig.show()


In [4]:
import os

scenarios = ["no_scenario", "demand_aware", "multi_objective", "peak_hour", "large_aircraft"]
throughput_idx = 13

for scenario in scenarios:
    base_path = f"BASELINE_observations_{scenario}.csv"
    ppo_path = f"PPO_observations_{scenario}.csv"
    dqn_path = f"DQN_observations_{scenario}.csv"
    odt_path = f"ODT_observations_{scenario}.csv"
    odt_finetuned_path = f"ODT_FINETUNED_observations_{scenario}.csv"

    if not (os.path.exists(base_path) and os.path.exists(ppo_path) and os.path.exists(dqn_path) and os.path.exists(odt_path) and os.path.exists(odt_finetuned_path)):
        print(f"Missing files for scenario '{scenario}', skipping.")
        continue

    obs_base = pd.read_csv(base_path).values
    obs_ppo = pd.read_csv(ppo_path).values
    obs_dqn = pd.read_csv(dqn_path).values
    obs_odt = pd.read_csv(odt_path).values
    obs_odt_finetuned = pd.read_csv(odt_finetuned_path).values

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=obs_base[:, throughput_idx], mode='lines', name='Baseline'))
    fig.add_trace(go.Scatter(y=obs_ppo[:, throughput_idx], mode='lines', name='PPO'))
    fig.add_trace(go.Scatter(y=obs_dqn[:, throughput_idx], mode='lines', name='DQN'))
    fig.add_trace(go.Scatter(y=obs_odt[:, throughput_idx], mode='lines', name='ODT'))
    fig.add_trace(go.Scatter(y=obs_odt_finetuned[:, throughput_idx], mode='lines', name='ODT Finetuned'))
    fig.update_layout(
        title=f"Average Throughput Over Time - {scenario}",
        xaxis_title='Step',
        yaxis_title='Average Throughput',
        legend_title='Agent',
        template='plotly_white'
    )
    fig.show()


In [5]:
import os

scenarios = ["no_scenario", "demand_aware", "multi_objective", "peak_hour", "large_aircraft"]
delay_idx = 14

for scenario in scenarios:
    base_path = f"BASELINE_observations_{scenario}.csv"
    ppo_path = f"PPO_observations_{scenario}.csv"
    dqn_path = f"DQN_observations_{scenario}.csv"
    odt_path = f"ODT_observations_{scenario}.csv"
    odt_finetuned_path = f"ODT_FINETUNED_observations_{scenario}.csv"

    if not (os.path.exists(base_path) and os.path.exists(ppo_path) and os.path.exists(dqn_path) and os.path.exists(odt_path) and os.path.exists(odt_finetuned_path)):
        print(f"Missing files for scenario '{scenario}', skipping.")
        continue

    obs_base = pd.read_csv(base_path).values
    obs_ppo = pd.read_csv(ppo_path).values
    obs_dqn = pd.read_csv(dqn_path).values
    obs_odt = pd.read_csv(odt_path).values
    obs_odt_finetuned = pd.read_csv(odt_finetuned_path).values

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=obs_base[:, delay_idx], mode='lines', name='Baseline'))
    fig.add_trace(go.Scatter(y=obs_ppo[:, delay_idx], mode='lines', name='PPO'))
    fig.add_trace(go.Scatter(y=obs_dqn[:, delay_idx], mode='lines', name='DQN'))
    fig.add_trace(go.Scatter(y=obs_odt[:, delay_idx], mode='lines', name='ODT'))
    fig.add_trace(go.Scatter(y=obs_odt_finetuned[:, delay_idx], mode='lines', name='ODT Finetuned'))
    fig.update_layout(
        title=f"Delay Over Time - {scenario}",
        xaxis_title='Step',
        yaxis_title='Delay (s)',
        legend_title='Agent',
        template='plotly_white'
    )
    fig.show()


In [6]:
import os

scenarios = ["no_scenario", "demand_aware", "multi_objective", "peak_hour", "large_aircraft"]
allocated_bw_idx = 7

for scenario in scenarios:
    base_path = f"BASELINE_observations_{scenario}.csv"
    ppo_path = f"PPO_observations_{scenario}.csv"
    dqn_path = f"DQN_observations_{scenario}.csv"
    odt_path = f"ODT_observations_{scenario}.csv"
    odt_finetuned_path = f"ODT_FINETUNED_observations_{scenario}.csv"

    if not (os.path.exists(base_path) and os.path.exists(ppo_path) and os.path.exists(dqn_path) and os.path.exists(odt_path) and os.path.exists(odt_finetuned_path)):
        print(f"Missing files for scenario '{scenario}', skipping.")
        continue

    obs_base = pd.read_csv(base_path).values
    obs_ppo = pd.read_csv(ppo_path).values
    obs_dqn = pd.read_csv(dqn_path).values
    obs_odt = pd.read_csv(odt_path).values
    obs_odt_finetuned = pd.read_csv(odt_finetuned_path).values

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=obs_base[:, allocated_bw_idx], mode='lines', name='Baseline'))
    fig.add_trace(go.Scatter(y=obs_ppo[:, allocated_bw_idx], mode='lines', name='PPO'))
    fig.add_trace(go.Scatter(y=obs_dqn[:, allocated_bw_idx], mode='lines', name='DQN'))
    fig.add_trace(go.Scatter(y=obs_odt[:, allocated_bw_idx], mode='lines', name='ODT'))
    fig.add_trace(go.Scatter(y=obs_odt_finetuned[:, allocated_bw_idx], mode='lines', name='ODT Finetuned'))
    fig.update_layout(
        title=f"Allocated Bandwidth Over Time - {scenario}",
        xaxis_title='Step',
        yaxis_title='Allocated Bandwidth',
        legend_title='Agent',
        template='plotly_white'
    )
    fig.show()


In [7]:
import os

scenarios = ["no_scenario", "demand_aware", "multi_objective", "peak_hour", "large_aircraft"]
allocated_demand_idx = 8

for scenario in scenarios:
    base_path = f"BASELINE_observations_{scenario}.csv"
    ppo_path = f"PPO_observations_{scenario}.csv"
    dqn_path = f"DQN_observations_{scenario}.csv"
    odt_path = f"ODT_observations_{scenario}.csv"
    odt_finetuned_path = f"ODT_FINETUNED_observations_{scenario}.csv"

    if not (os.path.exists(base_path) and os.path.exists(ppo_path) and os.path.exists(dqn_path) and os.path.exists(odt_path) and os.path.exists(odt_finetuned_path)):
        print(f"Missing files for scenario '{scenario}', skipping.")
        continue

    obs_base = pd.read_csv(base_path).values
    obs_ppo = pd.read_csv(ppo_path).values
    obs_dqn = pd.read_csv(dqn_path).values
    obs_odt = pd.read_csv(odt_path).values
    obs_odt_finetuned = pd.read_csv(odt_finetuned_path).values

    avg_allocated_to_demand_base = []
    avg_allocated_to_demand_ppo = []
    avg_allocated_to_demand_dqn = []
    avg_allocated_to_demand_odt = []
    avg_allocated_to_demand_odt_finetuned = []

    for i in range(len(obs_base)):
        avg_allocated_to_demand_base.append(np.mean(obs_base[:i, allocated_demand_idx]))
        avg_allocated_to_demand_ppo.append(np.mean(obs_ppo[:i, allocated_demand_idx]))
        avg_allocated_to_demand_dqn.append(np.mean(obs_dqn[:i, allocated_demand_idx]))
        avg_allocated_to_demand_odt.append(np.mean(obs_odt[:i, allocated_demand_idx]))
        avg_allocated_to_demand_odt_finetuned.append(np.mean(obs_odt_finetuned[:i, allocated_demand_idx]))

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=avg_allocated_to_demand_base, mode='lines', name='Baseline'))
    fig.add_trace(go.Scatter(y=avg_allocated_to_demand_ppo, mode='lines', name='PPO'))
    fig.add_trace(go.Scatter(y=avg_allocated_to_demand_dqn, mode='lines', name='DQN'))
    fig.add_trace(go.Scatter(y=avg_allocated_to_demand_odt, mode='lines', name='ODT'))
    fig.add_trace(go.Scatter(y=avg_allocated_to_demand_odt_finetuned, mode='lines', name='ODT Finetuned'))
    fig.update_layout(
        title=f"Allocated Demand Over Time - {scenario}",
        xaxis_title='Step',
        yaxis_title='Allocated Demand',
        legend_title='Agent',
        template='plotly_white'
    )
    fig.show()


/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



In [8]:
# Average Allocation to demand 
print("Baseline Average Allocation to Demand: ", np.mean(obs_base[:, 8]))
print("PPO Average Allocation to Demand: ", np.mean(obs_ppo[:, 8]))
print("DQN Average Allocation to Demand: ", np.mean(obs_dqn[:, 8]))
print("ODT Average Allocation to Demand: ", np.mean(obs_odt[:, 8]))
print("ODT Finetuned Average Allocation to Demand: ", np.mean(obs_odt_finetuned[:, 8]))    


Baseline Average Allocation to Demand:  0.4266234192714894
PPO Average Allocation to Demand:  0.39648292358707965
DQN Average Allocation to Demand:  0.41701801577083725
ODT Average Allocation to Demand:  0.4242388480885192
ODT Finetuned Average Allocation to Demand:  0.4245559648230204


In [9]:
# Average throughput across runs
print("Baseline Average Throughput Mbps: ", np.mean(obs_base[:, 13]))
print("PPO Average Throughput Mbps: ", np.mean(obs_ppo[:, 13]))
print("DQN Average Throughput Mbps: ", np.mean(obs_dqn[:, 13]))
print("ODT Average Throughput Mbps: ", np.mean(obs_odt[:, 13]))
print("ODT Finetuned Average Throughput Mbps: ", np.mean(obs_odt_finetuned[:, 13]))

Baseline Average Throughput Mbps:  35.93367489252425
PPO Average Throughput Mbps:  34.143247906647126
DQN Average Throughput Mbps:  35.278580538226144
ODT Average Throughput Mbps:  35.94617070904942
ODT Finetuned Average Throughput Mbps:  35.91880575610177


In [10]:
# Average Delay across runs
print("Baseline Average Delay ms: ", np.mean(obs_base[:, 14]))
print("PPO Average Delay ms: ", np.mean(obs_ppo[:, 14]))
print("DQN Average Delay ms: ", np.mean(obs_dqn[:, 14]))
print("ODT Average Delay ms: ", np.mean(obs_odt[:, 14]))
print("ODT Finetuned Average Delay ms: ", np.mean(obs_odt_finetuned[:, 14]))

Baseline Average Delay ms:  0.11816081702158318
PPO Average Delay ms:  0.11954713870328802
DQN Average Delay ms:  0.11954713870328802
ODT Average Delay ms:  0.11954713870328802
ODT Finetuned Average Delay ms:  0.11954713870328802


In [11]:
# Total Handovers 
print("Baseline Total Handovers: ", np.max(obs_base[:, 6]))
print("PPO Total Handovers: ", np.max(obs_ppo[:, 6]))
print("DQN Total Handovers: ", np.max(obs_dqn[:, 6]))
print("ODT Total Handovers: ", np.max(obs_odt[:, 6]))
print("ODT Finetuned Total Handovers: ", np.max(obs_odt_finetuned[:, 6]))

Baseline Total Handovers:  75.0
PPO Total Handovers:  64.0
DQN Total Handovers:  66.0
ODT Total Handovers:  68.0
ODT Finetuned Total Handovers:  68.0


In [12]:
import pandas as pd

scenarios = ["no_scenario", "demand_aware", "multi_objective", "peak_hour", "large_aircraft"]
agents = ["BASELINE", "PPO", "DQN", "ODT", "ODT_FINETUNED"]

# Column indices based on testscript output
allocated_idx = 7
ratio_idx = 8
demand_idx = 9
delay_idx = 11
throughput_idx = 12
handover_idx = 6
service_drop_idx = 16

rows = []
for scenario in scenarios:
    for agent in agents:
        path = f"{agent}_observations_{scenario}.csv"
        if not os.path.exists(path):
            continue
        data = pd.read_csv(path).values
        rows.append({
            "scenario": scenario,
            "agent": agent,
            "avg_allocated_MB": data[:, allocated_idx].mean(),
            "avg_allocation_ratio": data[:, ratio_idx].mean(),
            "avg_demand_MB": data[:, demand_idx].mean(),
            "avg_delay_s": data[:, delay_idx].mean(),
            "avg_throughput_mbps": data[:, throughput_idx].mean(),
            "total_handovers": data[:, handover_idx].max(),
            "total_service_drop_s": data[:, service_drop_idx].sum(),
        })

summary_df = pd.DataFrame(rows)
summary_df.sort_values(["scenario", "agent"], inplace=True)
summary_df


,scenario,agent,avg_allocated_MB,avg_allocation_ratio,avg_demand_MB,avg_delay_s,avg_throughput_mbps,total_handovers,total_service_drop_s
5,demand_aware,BASELINE,13.009212,0.974909,13.391065,3.838927,0.008779,75.0,240.0
7,demand_aware,DQN,12.996967,0.972984,13.407838,2.934130,0.008843,72.0,234.0
8,demand_aware,ODT,13.011914,0.973804,13.407838,2.927707,0.008834,71.0,229.0
9,demand_aware,ODT_FINETUNED,13.026088,0.974910,13.407838,3.838922,0.008793,71.0,232.0
6,demand_aware,PPO,12.934947,0.967478,13.407838,3.905374,0.008892,73.0,236.0
20,large_aircraft,BASELINE,17.557808,0.426623,43.169273,11.813000,0.008779,75.0,240.0
22,large_aircraft,DQN,17.153130,0.417018,43.149253,11.511493,0.008970,66.0,216.0
23,large_aircraft,ODT,17.414477,0.424239,43.149253,11.036331,0.008816,68.0,226.0
24,large_aircraft,ODT_FINETUNED,17.434884,0.424556,43.149253,11.880201,0.008793,68.0,226.0
21,large_aircraft,PPO,16.363452,0.396483,43.149253,79.020866,0.009720,64.0,206.0


In [13]:
# Metrics as rows; columns = (agent, scenario)
scenarios = ["no_scenario", "demand_aware", "multi_objective", "peak_hour", "large_aircraft"]
agents = ["BASELINE", "PPO", "DQN", "ODT", "ODT_FINETUNED"]
allocated_idx = 7
ratio_idx = 8
demand_idx = 9
delay_idx = 11
throughput_idx = 12
handover_idx = 6
service_drop_idx = 16
rows = []
for scenario in scenarios:
    for agent in agents:
        path = f"{agent}_observations_{scenario}.csv"
        if not os.path.exists(path):
            continue
        data = pd.read_csv(path).values
        rows.append({
            'scenario': scenario,
            'agent': agent,
            'avg_allocated_MB': data[:, allocated_idx].mean(),
            'avg_allocation_ratio': data[:, ratio_idx].mean(),
            'avg_demand_MB': data[:, demand_idx].mean(),
            'avg_delay_s': data[:, delay_idx].mean(),
            'avg_throughput_mbps': data[:, throughput_idx].mean(),
            'total_handovers': data[:, handover_idx].max(),
            'total_service_drop_s': data[:, service_drop_idx].sum(),
        })
summary_df = pd.DataFrame(rows)
long_df = summary_df.melt(id_vars=['agent', 'scenario'], var_name='metric', value_name='value')
pivot = long_df.pivot(index='metric', columns=['agent', 'scenario'], values='value')
pivot


agent,BASELINE,PPO,DQN,ODT,ODT_FINETUNED,BASELINE,PPO,DQN,ODT,ODT_FINETUNED,...,BASELINE,PPO,DQN,ODT,ODT_FINETUNED,BASELINE,PPO,DQN,ODT,ODT_FINETUNED
scenario,no_scenario,no_scenario,no_scenario,no_scenario,no_scenario,demand_aware,demand_aware,demand_aware,demand_aware,demand_aware,...,peak_hour,peak_hour,peak_hour,peak_hour,peak_hour,large_aircraft,large_aircraft,large_aircraft,large_aircraft,large_aircraft
metric,,,,,,,,,,,,,,,,,,,,,
avg_allocated_MB,11.717839,10.552646,11.556810,11.716599,11.689351,13.009212,12.934947,12.996967,13.011914,13.026088,...,11.717839,11.716599,11.716354,11.710621,11.710621,17.557808,16.363452,17.153130,17.414477,17.434884
avg_allocation_ratio,0.966714,0.874985,0.954337,0.966894,0.964879,0.974909,0.967478,0.972984,0.973804,0.974910,...,0.966714,0.966894,0.966871,0.966509,0.966509,0.426623,0.396483,0.417018,0.424239,0.424556
avg_delay_s,3.904943,51.758820,3.129468,3.903861,2.999424,3.838927,3.905374,2.934130,2.927707,3.838922,...,3.904943,3.903861,3.903979,2.987615,2.987615,11.813000,79.020866,11.511493,11.036331,11.880201
avg_demand_MB,12.298386,12.294173,12.294173,12.294173,12.294173,13.391065,13.407838,13.407838,13.407838,13.407838,...,12.298386,12.294173,12.294173,12.294173,12.294173,43.169273,43.149253,43.149253,43.149253,43.149253
avg_throughput_mbps,0.008779,0.010798,0.009059,0.008793,0.008834,0.008779,0.008892,0.008843,0.008834,0.008793,...,0.008779,0.008892,0.008917,0.008816,0.008816,0.008779,0.009720,0.008970,0.008816,0.008793
total_handovers,75.000000,61.000000,66.000000,73.000000,70.000000,75.000000,73.000000,72.000000,71.000000,71.000000,...,75.000000,73.000000,72.000000,72.000000,73.000000,75.000000,64.000000,66.000000,68.000000,68.000000
total_service_drop_s,240.000000,194.000000,216.000000,236.000000,227.000000,240.000000,236.000000,234.000000,229.000000,232.000000,...,240.000000,236.000000,234.000000,234.000000,236.000000,240.000000,206.000000,216.000000,226.000000,226.000000
